# Module 2: Hooks (~5 min)

Hooks observe every tool call — the same pattern NGS uses with CloudWatch to monitor every inference.

In this module we build a `DynastyAnalyticsHook` that:
- Resets per-turn counters at the start of each invocation
- Tracks which players and games have been queried
- Blocks repeat lookups (forcing the agent to reuse prior results)
- Warns when analysis depth gets high

In [ ]:
# install dependencies (run once)
%pip install strands-agents strands-agents-tools --quiet

import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

from strands import Agent
from strands.models import BedrockModel
from strands.hooks import HookProvider, HookRegistry, BeforeInvocationEvent, BeforeToolCallEvent

from dynasty_tools import lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff

## The DynastyAnalyticsHook

This hook implements two lifecycle callbacks:

| Event | Callback | Purpose |
| --- | --- | --- |
| `BeforeInvocationEvent` | `reset_turn` | Reset per-turn tool count, print session state |
| `BeforeToolCallEvent` | `track_and_gate` | Track queries, cancel repeat lookups |

The hook maintains a `queried_players` set and a `queried_games` set across turns.
When the agent tries to look up the same player or game week twice, the hook sets
`event.cancel_tool` with a message telling the model to reuse prior results.

In [ ]:
class DynastyAnalyticsHook(HookProvider):
    """Tracks analysis patterns and blocks repeat lookups.

    Like NGS CloudWatch monitoring every inference — this hook observes
    every tool call, tracks what's been queried, and prevents redundant lookups.
    """

    def __init__(self):
        self.queried_players: set = set()
        self.queried_games: set = set()
        self.tool_count: int = 0

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset_turn)
        registry.add_callback(BeforeToolCallEvent, self.track_and_gate)

    def reset_turn(self, event: BeforeInvocationEvent) -> None:
        """Reset per-turn counters. Keeps cross-turn memory of queried players."""
        self.tool_count = 0
        print(f"[ANALYTICS] \U0001f4ca Session state: {len(self.queried_players)} players queried, {len(self.queried_games)} games reviewed")

    def track_and_gate(self, event: BeforeToolCallEvent) -> None:
        """Track queries and block repeat lookups."""
        name = event.tool_use["name"]
        args = event.tool_use.get("input", {})
        self.tool_count += 1

        print(f"[ANALYTICS] \U0001f50d Tool call #{self.tool_count}: {name}({args})")

        # Track and gate player lookups
        if name == "lookup_player":
            player = args.get("player_name", "").lower()
            if player in self.queried_players:
                event.cancel_tool = (
                    f"Already looked up '{player}' this session. "
                    "Use the information from the previous lookup instead of repeating it."
                )
                print(f"[ANALYTICS] \U0001f6ab BLOCKED: repeat lookup for '{player}'")
                return
            self.queried_players.add(player)

        # Track game queries
        if name == "get_game_result":
            week = args.get("week", "")
            if week in self.queried_games:
                event.cancel_tool = (
                    f"Already reviewed week {week} this session. "
                    "Reference the earlier result instead of re-querying."
                )
                print(f"[ANALYTICS] \U0001f6ab BLOCKED: repeat game lookup for week {week}")
                return
            self.queried_games.add(week)

        # Depth warning
        if self.tool_count > 5:
            print(f"[ANALYTICS] \u26a0\ufe0f  High analysis depth: {self.tool_count} tools this turn")

## Create the agent with the hook

We attach the hook at agent creation via `hooks=[DynastyAnalyticsHook()]`.
The hook registers its callbacks automatically — no wiring needed beyond passing it in.

In [ ]:
hook = DynastyAnalyticsHook()

agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name="us-west-2"),
    tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
    hooks=[hook],
    callback_handler=None,
    system_prompt="""You are a 2004 New England Patriots dynasty analyst. Your approach mirrors
the best of patriots.com's coverage — evidence-first, narrative-aware.

When answering:
- Always look up the data before making claims. Never guess stats.
- Connect facts to story — why something happened matters as much as what happened.
- Be specific: cite game weeks, scores, stat lines.
- Describe players in terms of their role on the team.""",
)

print("Agent created with DynastyAnalyticsHook attached.")

## First query — watch the hook log

The `[ANALYTICS]` lines come from our hook. Watch the session state counter and the tool call tracker.

In [ ]:
result = agent("Tell me about Tom Brady")
print("\n---\n")
print(result)

## Same player again — watch the block

Now ask about Brady again. The hook remembers that `tom brady` is already in `queried_players`.
It will set `event.cancel_tool` which tells the agent framework to skip the tool call and
return the cancellation message to the model instead.

In [ ]:
result = agent("Look up Tom Brady again")
print("\n---\n")
print(result)

## What happened

The hook printed `BLOCKED: repeat lookup for 'tom brady'` and set `cancel_tool`.
The agent received the cancellation message instead of tool results, so it had to
synthesize from what it already knew. No wasted inference, no redundant API call.

This is the same principle NGS applies — you don't re-extract the same 11 features
for a play you've already processed. The hook enforces that discipline at the agent level.

In [ ]:
# try your own queries here
# - ask about a different player, then the same one again
# - ask about multiple players in one query to see tool_count climb
# - check hook.queried_players to see session state

print(f"Players queried this session: {hook.queried_players}")
print(f"Games reviewed this session: {hook.queried_games}")

## What's next

**Module 3** adds skills and steering — a dynasty-debate workflow skill plus a
fact-check guardrail that blocks claims without a prior lookup. The hook pattern
you just built is the observation layer; skills are the procedural knowledge layer.